# Generation of prompts for generating Dataset

## 1. imports 

In [39]:
import pandas as pd

## 2. Load sample data

In [40]:
baseline_df=pd.read_csv("../data/stock_prices/articles_gold_baseline_seed7.csv")
baseline_df

,obs_id,date,t_years,gold,silver,gas,oil,commodity_topics,price_gold,log_price_gold,dev_gold,price_silver,log_price_silver,dev_silver,label,IN
0,0,1970-01-01,0.000000,0,1,0,0,silver,NaN,NaN,NaN,1.556254,0.442282,0.0,0,0
1,1,1970-01-06,0.011905,1,0,1,0,"gas,gold",35.075149,3.557493,0.0,NaN,NaN,NaN,0,0
2,2,1970-01-09,0.023810,1,0,0,0,gold,34.758049,3.548411,0.0,NaN,NaN,NaN,0,0
3,3,1970-01-14,0.035714,0,1,1,1,"gas,oil,silver",NaN,NaN,NaN,1.537568,0.430202,0.0,0,0
4,4,1970-01-16,0.043651,1,0,0,0,gold,35.264961,3.562890,0.0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,2025-12-16,57.928571,0,1,0,0,silver,NaN,NaN,NaN,14.855160,2.698347,0.0,0,0
4996,4996,2025-12-18,57.936508,0,0,0,1,oil,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4997,4997,2025-12-23,57.948413,0,1,1,0,"gas,silver",NaN,NaN,NaN,15.012354,2.708873,0.0,0,0
4998,4998,2025-12-26,57.960317,0,1,1,0,"gas,silver",NaN,NaN,NaN,14.650034,2.684443,0.0,0,0


## 3. Article creation

### 3.1 template_gold_silver_struct_low_seed7_final_test


In [41]:
"""
Prompt template for generating synthetic financial news articles
about GOLD, SILVER, OIL, and GAS prices, conditioned on:
  - The article date
  - 21 days of price history ending on that date, for EACH sampled commodity
  - The set of 1-4 commodities the article discusses
  - The current price level of each commodity

(No break-referencing in this version -- break_in_window is always False,
break_date/break_magnitudes are always empty. The break-block parameters
are kept in the signature so build_prompt still works unchanged if break
support is reintroduced later.)

IMPORTANT: The generated articles NEVER name the commodities directly (no
"gold", "silver", "oil", "gas", "crude", "Brent", "LNG", etc.) and NEVER
state exact prices or percentage figures. Instead:
  - Each commodity is referred to via vague phrases ("the metal", "the fuel",
    "the commodity") or descriptive references to its use case.
  - Prices only shape the narrative's TONE (bullish/bearish/calm/volatile).

HARDENING:
  - DECOY MENTIONS: with probability `p_decoy`, the article may explicitly
    name 1-2 OTHER commodities (not in the sampled set) in passing.
  - AMBIGUOUS THEMES: with probability `p_ambiguous_theme`, one theme is
    drawn from a shared pool fitting several commodities.
  - VARIABLE SIGNAL COUNT: number of themes per commodity is sampled.

Usage:
    from prompt_template import build_prompt
    prompt = build_prompt(date, commodities, prices_by_commodity,
                          break_in_window, break_date, break_magnitudes, seed)
"""

import numpy as np

# ── Reuters-21578 word count distribution ─────────────────────────
_REUTERS_MEAN = 135.8
_REUTERS_VAR  = 18332.6
_REUTERS_MIN  = 2
_REUTERS_MAX  = 1669
_LN_SIGMA = np.sqrt(np.log(1 + _REUTERS_VAR / _REUTERS_MEAN**2))
_LN_MU    = np.log(_REUTERS_MEAN) - 0.5 * _LN_SIGMA**2


def sample_n_words(rng: np.random.Generator,
                   word_counts: list[int] | np.ndarray | None = None) -> int:
    """Sample a realistic article length."""
    if word_counts is not None:
        return int(rng.choice(word_counts))
    n = int(np.round(rng.lognormal(mean=_LN_MU, sigma=_LN_SIGMA)))
    return int(np.clip(n, _REUTERS_MIN, _REUTERS_MAX))


# ── Commodity-specific metadata ───────────────────────────────────
COMMODITY_META = {
    "gold": {
        "unit":      "USD/oz",
        "price_col": "price_gold",
        "family":    "metal",
        "signals": [
            ("safe-haven demand amid geopolitical uncertainty",        1900, 9999),
            ("central bank reserve diversification",                   1900, 9999),
            ("inflation hedging by institutional investors",           1950, 9999),
            ("dollar weakness driving flows into hard assets",         1971, 9999),
            ("monetary policy uncertainty ahead of a rate decision",   1950, 9999),
        ],
        "forbidden": [
            '"gold"', '"XAU"', '"yellow metal"', '"bullion"', '"golden"',
        ],
        "indirect": [
            ('"the metal favored by central banks"',              1900, 9999),
            ('"the traditional store of value"',                  1900, 9999),
            ('"the metal long associated with monetary reserves"', 1900, 9999),
        ],
    },
    "silver": {
        "unit":      "USD/oz",
        "price_col": "price_silver",
        "family":    "metal",
        "signals": [
            ("surging photovoltaic panel demand",                          2006, 9999),
            ("electronics manufacturing consumption",                      1975, 9999),
            ("consumption by the photographic film industry",              1900, 2005),
            ("industrial output data from Asian economies",                1970, 9999),
            ("the ratio between two related precious metals",              1900, 9999),
            ("dual-use demand from both investors and manufacturers",      1900, 9999),
        ],
        "forbidden": [
            '"silver"', '"XAG"', '"white metal"', '"bullion"',
        ],
        "indirect": [
            ('"the metal used in solar panels"',                    2006, 9999),
            ('"the metal consumed by the photographic industry"',   1900, 2005),
            ('"the industrially consumed precious metal"',          1900, 9999),
            ('"the metal prized by both investors and manufacturers"', 1900, 9999),
        ],
    },
    "oil": {
        "unit":      "USD/bbl",
        "price_col": "price_oil",
        "family":    "fuel",
        "signals": [
            ("production quota deliberations within a major exporters' alliance", 1965, 9999),
            ("refinery utilization rates along the U.S. Gulf Coast",              1950, 9999),
            ("tanker traffic disruptions at a key maritime chokepoint",           1940, 9999),
            ("debate over releases from strategic government stockpiles",         1977, 9999),
            ("upstream drilling activity and rig counts in shale basins",         2009, 9999),
            ("output developments in the North Sea producing region",             1976, 9999),
            ("a pipeline outage in a major producing region",                     1940, 9999),
        ],
        "forbidden": [
            '"oil"', '"crude"', '"petroleum"', '"WTI"', '"Brent"',
            '"barrel"', '"barrels"', '"bbl"', '"black gold"', '"OPEC"',
        ],
        "indirect": [
            ('"the fuel that powers global transport"',                1920, 9999),
            ('"the commodity pumped from wells and shipped by tanker"', 1900, 9999),
            ('"the feedstock refined into transport fuels"',           1920, 9999),
        ],
    },
    "gas": {
        "unit":      "USD/MMBtu",
        "price_col": "price_gas",
        "family":    "fuel",
        "signals": [
            ("winter heating demand forecasts across Europe",                      1960, 9999),
            ("underground storage injection levels ahead of the cold season",      1960, 9999),
            ("seaborne exports of the fuel in super-chilled liquefied form",       1990, 9999),
            ("pipeline flow reductions from a major eastern supplier",             1975, 9999),
            ("power-sector fuel switching as utilities weigh generation costs",    1980, 9999),
            ("procurement by city utilities ahead of the heating season",          1930, 9999),
        ],
        "forbidden": [
            '"gas"', '"natural gas"', '"LNG"', '"methane"',
            '"Henry Hub"', '"TTF"', '"MMBtu"', '"gasoline"',
        ],
        "indirect": [
            ('"the fuel used for heating and power generation"',        1930, 9999),
            ('"the pipeline-delivered fuel"',                           1940, 9999),
            ('"the energy source stored underground ahead of winter"',  1960, 9999),
        ],
    },
}

_ALL_COMMODITIES = list(COMMODITY_META.keys())

SHARED_THEMES = {
    "metal": [
        ("broad strength across the precious metals complex",            1900, 9999),
        ("investor rotation between defensive assets and equities",      1950, 9999),
        ("exchange inventory drawdowns reported by major depositories",  1950, 9999),
        ("physical demand from Asian retail buyers",                     1950, 9999),
    ],
    "fuel": [
        ("energy market jitters ahead of the northern-hemisphere winter", 1950, 9999),
        ("shifts in hedge fund positioning across energy futures",        1990, 9999),
        ("questions over the pace of the global energy transition",       2015, 9999),
        ("weather-driven demand uncertainty across major consuming regions", 1930, 9999),
    ],
    "generic": [
        ("a firmer dollar weighing on assets priced in the U.S. currency", 1971, 9999),
        ("index rebalancing flows across the broader commodity complex",   1992, 9999),
        ("risk appetite shifting with global growth expectations",         1950, 9999),
        ("speculative positioning data from the futures market",           1965, 9999),
    ],
}


def _era_filter(themes: list[tuple[str, int, int]], year: int) -> list[str]:
    """Keep only themes plausible for the given article year."""
    return [t for (t, y_from, y_to) in themes if y_from <= year <= y_to]


DECOY_DISPLAY = {
    "gold":   "gold",
    "silver": "silver",
    "oil":    "crude oil",
    "gas":    "natural gas",
}


def build_prompt(
    article_date: str,
    commodities: list[str],
    prices_by_commodity: dict,
    break_in_window: bool = False,
    break_date: str | None = None,
    break_magnitudes: dict[str, float] | None = None,
    p_reference_break: float = 0.6,
    p_decoy: float = 0.5,
    max_decoys: int = 2,
    p_ambiguous_theme: float = 0.35,
    seed: int = None,
    word_counts: list[int] | np.ndarray | None = None,
    all_texts: list[str] | None = None,
) -> dict:
    """
    Build a prompt for generating a synthetic financial news article
    covering 1-4 commodities.

    Parameters
    ----------
    article_date         : str   — date of the article, e.g. "1987-10-20"
    commodities          : list  — 1-4 of "gold", "silver", "oil", "gas"
    prices_by_commodity  : dict  — {commodity: {"prices_21d": [...], "current_price": float}}
    break_in_window      : bool  — always False in this run (break-referencing
                                   disabled); kept for signature compatibility
    break_date           : str   — unused in this run
    break_magnitudes     : dict  — unused in this run
    p_reference_break    : float — unused while break_in_window is always False
    p_decoy              : float — probability of naming 1-2 OTHER commodities
    max_decoys           : int   — maximum number of decoy commodities named
    p_ambiguous_theme    : float — probability of a shared/ambiguous theme
    seed                 : int   — random seed for reproducibility
    word_counts          : list  — Reuters-21578 word count distribution
    all_texts            : list  — Reuters-21578 article texts for style examples

    Returns
    -------
    dict with keys: "prompt", "references_break", "system", "n_words",
                    "decoys_named", "themes"
    """
    commodities = [c.strip().lower() for c in commodities]
    if not commodities:
        raise ValueError("At least one commodity is required.")
    if len(commodities) > 4:
        raise ValueError("At most four commodities per article are supported.")
    for c in commodities:
        if c not in COMMODITY_META:
            raise ValueError(f"Unknown commodity '{c}'. Must be one of {_ALL_COMMODITIES}.")
        if c not in prices_by_commodity:
            raise ValueError(f"Missing price data for commodity '{c}' in prices_by_commodity.")

    break_magnitudes = break_magnitudes or {}

    rng     = np.random.default_rng(seed)
    n_words = sample_n_words(rng, word_counts=word_counts)

    references_break = False
    if break_in_window and break_magnitudes:
        references_break = bool(rng.random() < p_reference_break)

    # ── Style examples ──────────────────────────────────────────────
    if all_texts is not None and len(all_texts) >= 3:
        date_seed   = int(article_date.replace("-", "")) % (2**31)
        example_rng = np.random.default_rng(date_seed)
        idxs        = example_rng.choice(len(all_texts), size=3, replace=False)
        examples    = [all_texts[i] for i in idxs]
        others      = [c for c in _ALL_COMMODITIES if c not in commodities]
        contrast    = str(example_rng.choice(others)) if others else commodities[0]
        comms       = [commodities[0], contrast, commodities[0]]
        example_block = "\n\nSTYLE EXAMPLES (real articles for tone/format reference only — do NOT copy content):\n"
        for k, (text, comm) in enumerate(zip(examples, comms), 1):
            example_block += f"\n--- Example {k} (commodity: {comm}) ---\n{text.strip()}\n"
        example_block += "\n---\n"
    else:
        example_block = ""

    article_year = int(article_date[:4])

    # ── Per-commodity price features ─────────────────────────────────
    per_commodity_features = {}
    for c in commodities:
        meta          = COMMODITY_META[c]
        entry         = prices_by_commodity[c]
        prices_21d    = entry["prices_21d"]
        current_price = entry["current_price"]

        prices = np.array(prices_21d) if len(prices_21d) >= 2 else np.array([current_price, current_price])
        pct_change_21d = (prices[-1] / prices[0] - 1) * 100
        pct_change_5d  = (prices[-1] / prices[-6] - 1) * 100 if len(prices) >= 6 else None
        rolling_vol    = float(np.std(np.diff(np.log(prices))) * np.sqrt(252) * 100) if len(prices) >= 2 else 0.0
        price_min      = float(prices.min())
        price_max      = float(prices.max())
        trend          = "upward" if pct_change_21d > 0 else "downward"
        if meta["family"] == "fuel":
            vol_level = "elevated" if rolling_vol > 45 else "moderate" if rolling_vol > 25 else "low"
        else:
            vol_level = "elevated" if rolling_vol > 20 else "moderate" if rolling_vol > 12 else "low"

        per_commodity_features[c] = {
            "meta": meta, "prices_21d": prices_21d, "current_price": current_price,
            "pct_change_21d": pct_change_21d, "pct_change_5d": pct_change_5d,
            "rolling_vol": rolling_vol, "price_min": price_min, "price_max": price_max,
            "trend": trend, "vol_level": vol_level,
        }

    # ── Break block: always empty in this run, since break_in_window
    # ── is always False ──────────────────────────────────────────────
    break_block = ""

    # ── Narrative themes ───────────────────────────────────────────
    chosen_themes = []
    for c in commodities:
        signals = _era_filter(COMMODITY_META[c]["signals"], article_year)
        if signals:
            n_signals = int(rng.integers(1, 3))
            picked    = rng.choice(signals, size=min(n_signals, len(signals)), replace=False).tolist()
            chosen_themes.extend(picked)

    if rng.random() < p_ambiguous_theme:
        families    = {COMMODITY_META[c]["family"] for c in commodities}
        shared_pool = []
        for fam in families:
            shared_pool += SHARED_THEMES[fam]
        shared_pool += SHARED_THEMES["generic"]
        shared_pool = _era_filter(shared_pool, article_year)
        if shared_pool:
            chosen_themes.append(str(rng.choice(shared_pool)))

    signals_line = f"  Narrative themes to weave in : {', '.join(chosen_themes)}\n" if chosen_themes else ""

    # ── Decoy commodities ────────────────────────────────────────────
    target_set = set(commodities)
    decoy_pool = [c for c in _ALL_COMMODITIES if c not in target_set]
    decoys: list[str] = []
    if decoy_pool and rng.random() < p_decoy:
        n_decoys = int(rng.integers(1, min(max_decoys, len(decoy_pool)) + 1))
        families = [COMMODITY_META[c]["family"] for c in commodities]
        weights  = np.array([2.0 if COMMODITY_META[c]["family"] in families else 1.0
                             for c in decoy_pool])
        weights  = weights / weights.sum()
        decoys   = [str(d) for d in rng.choice(decoy_pool, size=n_decoys, replace=False, p=weights)]

    if decoys:
        decoy_names = [DECOY_DISPLAY[d] for d in decoys]
        decoy_line  = f"  Decoy commodities (may be NAMED, in passing only) : {', '.join(decoy_names)}\n"
        decoy_instruction = (
            "\n- DECOY MENTIONS: You MAY — and briefly should — explicitly name the following "
            f"OTHER commodit{'ies' if len(decoy_names) > 1 else 'y'}: {', '.join(decoy_names)}. "
            "Use them only in passing (one or two sentences each at most): a comparison of how "
            "they traded relative to the unnamed subjects, spillovers between the markets, or a "
            "brief aside. Invent their behavior freely. They must remain PERIPHERAL — the "
            "article's subject is still the unnamed commodities described above, and the naming "
            "ban applies to all subject commodities at all times, including inside these comparisons."
        )
    else:
        decoy_line        = ""
        decoy_instruction = ""

    # ── Naming rules ──────────────────────────────────────────────────
    forbidden, indirect = [], []
    for c in commodities:
        meta = COMMODITY_META[c]
        forbidden += meta["forbidden"]
        indirect  += _era_filter(meta["indirect"], article_year)

    families = {COMMODITY_META[c]["family"] for c in commodities}
    if len(families) == 1:
        fam = next(iter(families))
        generic_terms = ['"the metal"', '"the precious metal in question"'] if fam == "metal" \
            else ['"the fuel"', '"the energy commodity in question"']
    else:
        generic_terms = ['"the commodity"', '"the asset"']

    naming_instruction = (
        "\n- CRITICAL: Never name any of the subject commodities directly. Do NOT write any of "
        "the following words, nor any other direct synonym, ticker, benchmark, or "
        f"producer-cartel name: {', '.join(forbidden)}. "
        "Also avoid each commodity's standard quoting unit. "
        "Instead use vague or generic phrases such as "
        f"{', '.join(generic_terms)}, \"the commodity\", \"the asset\", "
        "or describe each one indirectly through its use case "
        f"(e.g. {' or '.join(indirect[:2])})). "
        "The reader should be able to guess which commodities these are only through context "
        "and the narrative themes, never through an explicit name."
        + (" The ONLY names you may write are those of the decoy commodities listed "
           "below — and even then, only as peripheral comparisons, never as a subject."
           if decoys else "")
    )

    price_instruction = (
        "\n- CRITICAL: Do NOT state exact prices, percentage changes, or numeric figures "
        "anywhere in the article (no \"$34.76\", no \"1.03%\", no specific numbers). "
        "The price data provided above is for YOUR understanding only, to inform the "
        "tone and direction of the narrative — e.g. whether the mood is bullish, "
        "bearish, calm, or volatile. Give the reader only a vague GLIMPSE of what might "
        "have happened to prices through qualitative language "
        "(\"edged higher\", \"came under pressure\", \"traded without much conviction\"), "
        "never through concrete figures."
    )

    subjects_str = ", ".join(commodities)
    consistency_instruction = (
        "\n- LABEL FIDELITY (most important): read as a whole, the article must be "
        f"unambiguously about the subject commodities given in the context ({subjects_str}) — "
        "even though none are named. Each subject's narrative themes must be present in the "
        "article; every invented event must be the kind of event that would move these "
        "commodities' markets. A well-informed reader must be able to infer all subjects "
        "correctly from context alone"
        + (", and must NOT be led to conclude the article is mainly about a decoy — "
           "decoys are contrast, never a competing subject." if decoys else ".")
        + "\n- TONE-DATA CONSISTENCY: the qualitative price language for each subject must match "
        "its trend and volatility data above in direction and intensity."
        + f"\n- ERA CONSISTENCY: everything must be plausible for {article_date} — no "
        "technologies, market structures, institutions, or vocabulary that did not "
        "exist yet at that date, and nothing that had clearly disappeared by then."
    )

    system = (
        "You are a financial journalist writing for a commodity markets newswire. "
        "Your articles are professional in tone and written as of the publication date "
        "— you only know what has happened up to and including that date. "
        "Your primary job is to describe the EVENTS and CAUSES driving the commodity "
        "market — political decisions, economic developments, supply disruptions, "
        "central bank actions, trade policy changes, geopolitical tensions. "
        "You never name the specific commodities your article is about — you refer "
        "to them only through vague phrases or descriptive references to their use case. "
        "Other, unrelated commodities may be named explicitly, but only when the "
        "instructions list them as permitted decoy comparisons, and only in passing. "
        "You never state exact prices or percentage figures — price information only "
        "shapes the tone of the narrative (bullish/bearish/calm/volatile), never as "
        "concrete numbers. "
        "You may reference real institutions (e.g. Federal Reserve, LBMA, CME Group, "
        "the IEA, the EIA), real countries, and real named individuals. "
        "However, all specific events, decisions, announcements, and incidents described "
        "in the article must be entirely invented — do not reference or reproduce any "
        "real historical events."
    )

    multi_instruction = (
        f"- This article covers {len(commodities)} related commodities — discuss each of them, "
        "their relationships, and how they moved relative to one another, "
        "without naming any of them directly."
        if len(commodities) > 1 else ""
    )

    price_blocks = []
    for c in commodities:
        f    = per_commodity_features[c]
        meta = f["meta"]
        name = c.capitalize()
        pct5 = f"{f['pct_change_5d']:+.2f}%" if f['pct_change_5d'] is not None else 'n/a'
        block = (
            f"  {name} current price    : ${f['current_price']:.2f} ({meta['unit']})  (as of {article_date})\n"
            f"  {name} 21-day trend     : {f['trend']} ({f['pct_change_21d']:+.2f}% over the window)\n"
            f"  {name} 5-day trend      : {pct5}\n"
            f"  {name} price range      : ${f['price_min']:.2f} - ${f['price_max']:.2f} ({meta['unit']}) over the past 21 days\n"
            f"  {name} volatility       : {f['vol_level']} ({f['rolling_vol']:.1f}% annualized)\n"
            f"  {name} price history    : {[round(p, 2) for p in f['prices_21d']]}  (oldest → most recent, {meta['unit']})"
        )
        price_blocks.append(block)
    prices_block = "\n".join(price_blocks)

    # ── Always empty in this run ─────────────────────────────────────
    break_line = ""

    prompt = f"""Write a financial news article about a commodity market,
published on {article_date}.
{example_block}
CONTEXT — for YOUR understanding only, do not state these figures in the article:
  Commodities (internal only)  : {subjects_str}
{signals_line}{decoy_line}{prices_block}
{break_line}

{break_block}

INSTRUCTIONS:
- The article must be PRIMARILY about invented events and causes that explain the
  market mood — political decisions, economic data, supply shocks, central bank
  signals, trade policy, geopolitical tensions, etc.
- Write as of {article_date}: only reference events up to this date.
- Invent all specific events and causes. You may name real institutions, countries,
  and individuals — but the specific events must be invented.
- Do NOT reference or reproduce any real historical events.
- Do NOT use future tense or make price predictions.
{multi_instruction}{naming_instruction}{decoy_instruction}{price_instruction}{consistency_instruction}
- Keep the article to approximately {n_words} words.
- Start with the headline, then the body. No preamble.
"""

    return {
        "system":           system,
        "prompt":           prompt.strip(),
        "references_break": references_break,
        "n_words":          n_words,
        "decoys_named":     decoys,
        "themes":           chosen_themes,
    }

In [42]:
import asyncio
import json
import os
from pathlib import Path

import aiohttp
import nest_asyncio
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.asyncio import tqdm_asyncio

MODEL        = "google/gemini-3.1-flash-lite"
MAX_TOKENS   = 1000
MAX_PARALLEL = 10
RETRY_LIMIT  = 3
RETRY_DELAY  = 2.0

PRICE_COLS = {c: f"price_{c}" for c in ("gold", "silver", "oil", "gas")}


def _load_api_key():
    try:
        env_path = Path(__file__).parent.parent / ".env"
    except NameError:
        env_path = Path.cwd().parent / ".env"
    load_dotenv(dotenv_path=env_path)
    key = os.getenv("GEMINI_API_KEY")
    if not key:
        raise ValueError("GEMINI_API_KEY not found in .env")
    return key


def _load_baseline(csv_path):
    df = pd.read_csv(csv_path, parse_dates=["date"])
    return df.sort_values("date").reset_index(drop=True)


def _get_prices(df, date, price_col):
    row_idx = df.index[df["date"] == pd.Timestamp(date)]
    if len(row_idx) == 0:
        raise ValueError(f"Date {date} not found in baseline CSV")
    idx           = row_idx[0]
    start         = max(0, idx - 21)
    prices_21d    = df[price_col].iloc[start: idx].tolist()
    current_price = float(df[price_col].iloc[idx])
    return prices_21d, current_price


async def _call_api(session, api_key, prompt_result):
    url     = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {api_key}"}
    payload = {
        "model": MODEL, "max_tokens": MAX_TOKENS,
        "messages": [
            {"role": "system", "content": prompt_result["system"]},
            {"role": "user",   "content": prompt_result["prompt"]},
        ],
    }
    for attempt in range(RETRY_LIMIT):
        try:
            async with session.post(url, headers=headers, json=payload) as resp:
                if resp.status == 429:
                    await asyncio.sleep(RETRY_DELAY * (attempt + 1)); continue
                resp.raise_for_status()
                data = await resp.json()
                return data["choices"][0]["message"]["content"].strip()
        except Exception as e:
            if attempt < RETRY_LIMIT - 1: await asyncio.sleep(RETRY_DELAY)
            else: raise e
    raise RuntimeError("All retries exhausted")


async def _process_row(session, api_key, row, baseline_df, semaphore,
                       build_prompt, word_counts, all_texts):
    async with semaphore:
        date_str = str(row["date"])[:10]
        obs_id   = int(row["obs_id"])

        # ── one-hot commodities, read directly from sample_df ──────────
        commodities_onehot = {c: int(row[c]) for c in ("gold", "silver", "oil", "gas")}
        parts = [c for c, v in commodities_onehot.items() if v == 1]

        if not parts:
            raise ValueError(f"obs_id {obs_id}: no commodity flagged (all zero)")

        # ── price history for every sampled commodity ─────────────────
        prices_by_commodity = {}
        for c in parts:
            p21, cur = _get_prices(baseline_df, date_str, PRICE_COLS[c])
            prices_by_commodity[c] = {"prices_21d": p21, "current_price": cur}

        # ── no break-referencing in this simplified run ────────────────
        break_in_window  = False
        break_date       = None
        break_magnitudes = {}

        prompt_result = build_prompt(
            article_date         = date_str,
            commodities           = parts,
            prices_by_commodity   = prices_by_commodity,
            break_in_window       = break_in_window,
            break_date            = break_date,
            break_magnitudes      = break_magnitudes,
            seed                  = obs_id,
            word_counts           = word_counts,
            all_texts             = all_texts,
        )

        raw      = await _call_api(session, api_key, prompt_result)
        lines    = [l for l in raw.split("\n") if l.strip()]
        headline = lines[0].strip() if lines else ""
        body     = "\n".join(lines[1:]).strip() if len(lines) > 1 else ""

        return {
            "metadata": {
                "obs_id":           obs_id,
                "article_date":     date_str,
                "commodities":      commodities_onehot,
                "commodity":        " and ".join(parts),
                "prices": {
                    c: {"current_price": round(d["current_price"], 4),
                        "prices_21d":    [round(p, 4) for p in d["prices_21d"]]}
                    for c, d in prices_by_commodity.items()
                },
                "references_break": prompt_result["references_break"],
                "n_words_target":   prompt_result["n_words"],
                "themes":           prompt_result["themes"],
                "decoys_named":     prompt_result["decoys_named"],
                "model":            MODEL,
            },
            "headline": headline,
            "body":     body,
        }


async def _generate_all_async(sample_df, baseline_csv, out_path,
                               build_prompt, word_counts, all_texts):
    api_key     = _load_api_key()
    baseline_df = _load_baseline(baseline_csv)
    semaphore   = asyncio.Semaphore(MAX_PARALLEL)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    async with aiohttp.ClientSession(
            connector=aiohttp.TCPConnector(limit=MAX_PARALLEL)) as session:
        tasks = [
            _process_row(session, api_key, row, baseline_df, semaphore,
                         build_prompt, word_counts, all_texts)
            for _, row in sample_df.iterrows()
        ]
        results = await tqdm_asyncio.gather(*tasks, desc="Generating articles")

    articles = sorted(results, key=lambda x: x["metadata"]["obs_id"])

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(articles)} articles → {out_path}")
    return articles


def generate_all(sample_df, dataset_csv, baseline_csv, build_prompt,
                 out_path=None, word_counts=None, all_texts=None):
    if out_path is None:
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = f"{Path(dataset_csv).stem}_{timestamp}.json"
        out_path  = str(Path(dataset_csv).parent.parent / "articles" / filename)
    nest_asyncio.apply()
    return asyncio.run(_generate_all_async(
        sample_df, baseline_csv, out_path, build_prompt, word_counts, all_texts))

In [ ]:
# BASELINE_CSV = "../data/stock_prices/articles_all_commodities_baseline_seed7.csv"
# N_ARTICLES   = 200   # raised from 20 so each of the 15 groups still gets 2 articles
# SEED         = 42

# from itertools import combinations

# ALL_COMMODITIES = ["gold", "silver", "oil", "gas"]

# # All possible non-empty subsets: 4 singles + 6 pairs + 4 triples + 1 quadruple = 15 groups
# COMBOS = [
#     combo
#     for r in range(1, len(ALL_COMMODITIES) + 1)
#     for combo in combinations(ALL_COMMODITIES, r)
# ]
# print(f"{len(COMBOS)} groups:", COMBOS)

# baseline_df = (pd.read_csv(BASELINE_CSV, parse_dates=["date"])
#                  .sort_values("date").reset_index(drop=True))

# missing = [c for c in ["price_gold", "price_silver", "price_oil", "price_gas"]
#            if c not in baseline_df.columns]
# if missing:
#     raise ValueError(f"Baseline CSV is missing price columns: {missing}")

# valid_dates = baseline_df["date"].iloc[21:].reset_index(drop=True)

# rng = np.random.default_rng(SEED)
# base, rem = divmod(N_ARTICLES, len(COMBOS))
# counts = {combo: base for combo in COMBOS}
# for combo in [COMBOS[i] for i in rng.permutation(len(COMBOS))[:rem]]:
#     counts[combo] += 1

# rows, obs_id = [], 0
# for combo in COMBOS:
#     for d in rng.choice(valid_dates, size=counts[combo], replace=False):
#         rows.append({
#             "obs_id": obs_id,
#             "date":   str(pd.Timestamp(d))[:10],
#             "gold":   int("gold"   in combo),
#             "silver": int("silver" in combo),
#             "oil":    int("oil"    in combo),
#             "gas":    int("gas"    in combo),
#         })
#         obs_id += 1

# sample_df = pd.DataFrame(rows)
# print(sample_df[["gold", "silver", "oil", "gas"]].sum())

# articles = generate_all(
#     sample_df    = sample_df,
#     dataset_csv  = BASELINE_CSV,
#     baseline_csv = BASELINE_CSV,
#     build_prompt = build_prompt,
#     word_counts  = None,
#     all_texts    = None,
# )

15 groups: [('gold',), ('silver',), ('oil',), ('gas',), ('gold', 'silver'), ('gold', 'oil'), ('gold', 'gas'), ('silver', 'oil'), ('silver', 'gas'), ('oil', 'gas'), ('gold', 'silver', 'oil'), ('gold', 'silver', 'gas'), ('gold', 'oil', 'gas'), ('silver', 'oil', 'gas'), ('gold', 'silver', 'oil', 'gas')]
gold      106
silver    105
oil       106
gas       107
dtype: int64


Generating articles: 100%|██████████| 200/200 [00:32<00:00,  6.08it/s]


Saved 200 articles → ../data/articles/articles_all_commodities_baseline_seed7_20260722_185151.json


In [47]:
BASELINE_CSV = "../data/stock_prices/articles_all_commodities_baseline_seed7.csv"

baseline_df = (pd.read_csv(BASELINE_CSV, parse_dates=["date"])
                 .sort_values("date").reset_index(drop=True))

missing = [c for c in ["price_gold", "price_silver", "price_oil", "price_gas"]
           if c not in baseline_df.columns]
if missing:
    raise ValueError(f"Baseline CSV is missing price columns: {missing}")

# The whole dataset, no sampling/balancing at all -- use every row as-is.
sample_df = baseline_df[["obs_id", "date", "gold", "silver", "oil", "gas"]].copy()
sample_df["date"] = sample_df["date"].dt.strftime("%Y-%m-%d")

n_flagged = sample_df[["gold", "silver", "oil", "gas"]].sum(axis=1)
n_empty   = (n_flagged == 0).sum()
if n_empty:
    print(f"Warning: {n_empty} rows have no commodity flagged -- dropping them.")
    sample_df = sample_df[n_flagged > 0].reset_index(drop=True)

print(f"Total articles to generate: {len(sample_df)}")
print(sample_df[["gold", "silver", "oil", "gas"]].sum())

combo_counts = sample_df.apply(
    lambda r: "+".join(c for c in ("gold","silver","oil","gas") if r[c] == 1), axis=1
).value_counts()
print("\nCombination distribution:")
print(combo_counts)

articles = generate_all(
    sample_df    = sample_df,
    dataset_csv  = BASELINE_CSV,
    baseline_csv = BASELINE_CSV,
    build_prompt = build_prompt,
    word_counts  = None,
    all_texts    = None,
)

Total articles to generate: 5000
gold      2958
silver    2106
oil       1609
gas       1623
dtype: int64

Combination distribution:
gold                   1178
silver                  614
gold+silver             482
gas                     433
oil                     413
gold+oil                312
gold+gas                307
gold+silver+oil         208
silver+gas              190
gold+silver+gas         187
silver+oil              170
gold+silver+oil+gas     170
oil+gas                 137
gold+oil+gas            114
silver+oil+gas           85
Name: count, dtype: int64


Generating articles: 100%|██████████| 5000/5000 [15:00<00:00,  5.55it/s]



Saved 5000 articles → ../data/articles/articles_all_commodities_baseline_seed7_20260722_190748.json
